# A-side masking on Qwen3-8B (same design as the 4B pilot)

Pick an **A100**, then *Run all*. On Colab, keep the tab open. Off Colab (any machine with an A100-class GPU, 40 GB+), run it with Jupyter or `papermill colab_mask_8b.ipynb out.ipynb`; the code and results go under your home directory.

Runs the masking code exactly as the 4B pilot ran it (commit `f4914e4`), on the same 200 source questions: 100 two-fact toy and 100 two-hop MuSiQue, seed 0.

Stages: code and install, benchmarks, load last night's 8B B1 / B1-mask rows (for the endpoint check), smoke (2 questions per cell), three composition probes on the 200 sources, then the five mask conditions on the questions that require composition. Every answer is saved as it finishes; *Run all* again on the same runtime resumes. The tarball downloads at the end.

In [ ]:
MODEL, DTYPE = "Qwen/Qwen3-8B", "bfloat16"
SEEDS, N_PER_CELL = [0], 100
REPO = "https://github.com/JoshBong/latent-dispatch.git"
COMMIT = "f4914e4"                        # masking code as run for the 4B pilot
ARCHIVE = "Qwen3-8B-bfloat16-2026-11-14"   # 8B B1 / B1-mask rows, read from main
import os
BASE = "/content" if os.path.isdir("/content") else os.path.expanduser("~")   # Colab, or any GPU machine
ROOT = f"{BASE}/latent-dispatch-mask8b"

In [ ]:
import json, os, subprocess, sys
import torch

def sh(cmd):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if r.stdout.strip():
        print(r.stdout[-(6000 if r.returncode else 1500):].strip(), flush=True)
    if r.returncode:
        raise RuntimeError(f"failed: {cmd}")

assert torch.cuda.is_available(), "no GPU: pick an A100"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(torch.cuda.get_device_name(0), f"{gb:.0f} GB")
assert gb > 30, "Qwen3-8B needs an A100 (a T4 or L4 is too small)"
if not os.path.exists(ROOT):
    sh(f"rm -rf {ROOT}.tmp && git clone -q {REPO} {ROOT}.tmp && git -C {ROOT}.tmp checkout -q {COMMIT} && mv {ROOT}.tmp {ROOT}")
os.chdir(ROOT)
sh("git log -1 --format='running commit %h (%s)'")
sh("pip install -q -e '.[dev]' 'papermill>=2.6'")
sys.path.insert(0, ROOT)

In [ ]:
names = ["toy", "toy_d0", "musique", "hotpotqa"]
missing = [n for n in names if not os.path.exists(f"benchmark/{n}.jsonl")]
if missing:
    sh("python -m latent_dispatch.data " + " ".join(missing))
sh("cd benchmark && sha256sum -c SHA256SUMS")

In [ ]:
# Last night's 8B B1 and B1-mask rows, so the masking notebook can check full = B1 and blocked = B1-mask.
# Not needed for the run itself: if this fails, the run goes on without that check.
from latent_dispatch import results
try:
    if not os.path.exists(f"archive/{ARCHIVE}.tar.gz"):
        sh(f"git show origin/main:archive/{ARCHIVE}.tar.gz > archive/{ARCHIVE}.tar.gz")
    results.load_archive(f"archive/{ARCHIVE}.tar.gz")
except Exception as e:
    print("!!! could not load the 8B archive; endpoint check against B1 / B1-mask will be missing:", e)

In [ ]:
# Smoke: 2 questions per cell through every mask condition. Stops here if anything is broken.
import logging
import papermill as pm
log = logging.getLogger("papermill"); log.setLevel(logging.INFO); log.propagate = False
if not log.handlers:
    log.addHandler(logging.StreamHandler(sys.stdout))
os.makedirs("notebooks/runs", exist_ok=True)
tag = MODEL.split("/")[-1]
pm.execute_notebook("notebooks/h1_masking.ipynb", f"notebooks/runs/{tag}-h1_masking-smoke.ipynb", kernel_name="python3",
                    log_output=True, parameters=dict(MODEL=MODEL, DTYPE=DTYPE, SMOKE=True, SEEDS=SEEDS,
                                                     N_PER_CELL=N_PER_CELL, SKIP=[]))

In [ ]:
# Composition probes on the 200 sources: closed book, and each gold paragraph alone. A source "requires
# composition" when all three are wrong. Same rule and scope as the 4B pilot.
import gc
from latent_dispatch import arms, data
from latent_dispatch.model import Model

qs = data.per_cell(data.load("toy", seed=SEEDS, qtype="hop1", placement="AB")
                   + data.load("musique", seed=SEEDS, qtype="2hop", placement="AB"), N_PER_CELL)
assert len({(q["dataset"], q["source_id"]) for q in qs}) == len(qs)
PROBES = {"b0": (arms.b0, {}),
          "b1_only_hop1": (arms.b1, {"keep_gold_hops": [0]}),
          "b1_only_hop2": (arms.b1, {"keep_gold_hops": [1]})}
todo = 0
for name, (arm, options) in PROBES.items():
    path = results.results_path(MODEL, DTYPE, name)
    saved = results._read(path)
    results._check(path, saved, {"arm": arm.__name__, "options": options})
    todo += sum(q["id"] not in saved for q in qs)
print(todo, "probe answers to run")
if todo:
    model = Model.load(MODEL, dtype=DTYPE)
    for name, (arm, options) in PROBES.items():
        results.run(model, qs, name, arm, **options)
    del model; gc.collect(); torch.cuda.empty_cache()
df = results.load_results(qs, PROBES, MODEL, DTYPE)
labels = {}
for q in qs:
    found = df[df.id == q["id"]]
    assert len(found) == len(PROBES), f"missing a probe for {q['id']}"
    labels[f"{q['dataset']}:{q['source_id']}"] = bool((found.em == 0).all())
folder = results.results_path(MODEL, DTYPE, "x").parent
(folder / "part0_requires_composition.json").write_text(json.dumps(
    {"seeds": SEEDS, "smoke": False, "scope": "masking_only", "n_per_cell": N_PER_CELL, "labels": labels}, indent=2) + "\n")
for ds in ("toy_d2", "musique"):
    k = [v for key, v in labels.items() if key.startswith(ds + ":")]
    print(ds, sum(k), "of", len(k), "sources require composition")

In [ ]:
# The masking run. Packs and downloads even if the notebook stops partway, so finished rows are not lost.
import glob
err = None
try:
    pm.execute_notebook("notebooks/h1_masking.ipynb", f"notebooks/runs/{tag}-h1_masking.ipynb", kernel_name="python3",
                        log_output=True, parameters=dict(MODEL=MODEL, DTYPE=DTYPE, SMOKE=False, SEEDS=SEEDS,
                                                         N_PER_CELL=N_PER_CELL, SKIP=[]))
except Exception as e:
    err = e
    print("\n!!! masking notebook stopped:", str(e)[-800:], flush=True)
out = f"{BASE}/results-{tag}-{DTYPE}-masking.tar.gz"
sh(f"tar czf {out} results/{tag}-{DTYPE} " + " ".join(sorted(glob.glob(f"notebooks/runs/{tag}-*.ipynb"))))
try:
    from google.colab import files
    files.download(out)
except ImportError:
    print("not on Colab: copy the tarball from", out)
if err:
    raise err
print("done:", out)